# miniLLM v2.3 — préparation des données de pré-entraînement (5 milliards de tokens, sources variées)

Construit `data/v23/stable/` (4,25 Md de tokens) et `data/v23/decay/` (0,75 Md : texte moderne de qualité, conversations multi-tours, questions-réponses). Détail et justification : `docs/V2_3_PLAN.md`.

**Aucun GPU nécessaire.** C'est surtout du réseau et de l'écriture sur le Drive : prévois plusieurs sessions. La commande est **reprenable** : si la session coupe, relance la cellule 3, elle reprend exactement où elle s'était arrêtée.

⚠️ Avant la cellule 3, lis le résultat de la cellule 2 (`inspect`) : les noms de colonnes et l'organisation des fichiers de certaines sources n'ont pas pu être vérifiés hors ligne.

In [ ]:
# ── 1) Drive, code (branche v2.3), plan ──
import os, sys, glob, json, shutil
from google.colab import drive
drive.mount("/content/drive")
cands = (["/content/drive/MyDrive/MiniLLM_v2"] + glob.glob("/content/drive/MyDrive/*/MiniLLM_v2") + glob.glob("/content/drive/Shareddrives/*/MiniLLM_v2"))
BASE = next((c for c in cands if os.path.isdir(f"{c}/data")), None)
assert BASE, "Dossier MiniLLM_v2 introuvable"
DATA = f"{BASE}/data"
OUT = f"{DATA}/v23"
BUDGET = 5e9               # tokens au total (mêmes proportions si tu changes : 3e9 pour un essai, 10e9 plus tard)
CODE_DIR = "/content/minillm_v2"
!pip -q install tokenizers datasets
if os.path.exists(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} fetch -q origin v2.3 && git -C {CODE_DIR} checkout -q v2.3 && git -C {CODE_DIR} pull -q --ff-only
else:
    !git clone -q -b v2.3 https://github.com/bono-p/minillm_v2.git {CODE_DIR}
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
import shutil
tot, used, free = shutil.disk_usage(BASE)
print(f"Drive : {free / 1e9:.1f} Go libres sur {tot / 1e9:.1f} Go")
!python prepare_mix.py plan --budget {BUDGET}


In [ ]:
# ── 2) INSPECTION (réseau, ~2 min) : colonnes et premier exemple de chaque source ──
!python prepare_mix.py inspect --n 2
# Vérifie : (a) que les colonnes utilisées par le plan existent (language, collection, text, conversations…) ;
#           (b) que les filtres du plan (language=French, collection=OpenCulture…) correspondent aux valeurs réellement présentes.
# Si une valeur diffère (ex. « fr » au lieu de « French »), dis-le moi : on corrige mixplan.py, rien d'autre.


In [ ]:
# ── 3) CONSTRUCTION (reprenable, longue) ──
EXISTING = f"{DATA}/pretrain"            # corpus v2.2 déjà tokenisé : train.bin, val.bin, meta.json
assert os.path.exists(f"{EXISTING}/train.bin"), f"{EXISTING}/train.bin introuvable"
!python prepare_mix.py build --out {OUT} --tokenizer {DATA}/tokenizer.json --existing {EXISTING} --budget {BUDGET}


In [ ]:
# ── 4) Bilan : proportions réellement obtenues, répétitions, documents rejetés ──
!python prepare_mix.py status --out {OUT}
if os.path.exists(f"{OUT}/mix_report.json"):
    r = json.load(open(f"{OUT}/mix_report.json"))
    print("\nTotal :", f"{r['total_tokens']:,}", "tokens")
    print("Parts par catégorie :", r["parts_par_categorie"])


**À m'envoyer :** la sortie de la cellule 2 (inspection) avant de lancer la cellule 3, puis le bilan de la cellule 4.